In [ ]:
import json

# --- ECCEZIONI PERSONALIZZATE ---

class ErroreValidazione(Exception):
    """Eccezione base per gli errori di validazione."""
    pass

class ErroreDatiMancanti(ErroreValidazione):
    """Lanciata quando manca una chiave obbligatoria o un elemento nel JSON."""
    pass

class ErroreTipoNonValido(ErroreValidazione):
    """Lanciata quando un campo ha un tipo non valido (es. numero negativo o stringa al posto di lista)."""
    pass


# --- FUNZIONI PER LE SINGOLE REGOLE ---

def verifica_documenti_richiesti(sinistro: dict, requisito: dict) -> dict:
    """
    Regole 1 & 6: Verifica il tipo di documenti forniti e controlla se tutti i documenti richiesti sono presenti.
    """
    if "documents" not in sinistro or not isinstance(sinistro["documents"], list):
        raise ErroreDatiMancanti("Il sinistro non contiene una lista valida di 'documents'.")
    if "required_documents" not in requisito or not isinstance(requisito["required_documents"], list):
        raise ErroreDatiMancanti("Il requisito non contiene una lista valida di 'required_documents'.")

    documenti_forniti = set(sinistro["documents"])
    documenti_richiesti = set(requisito["required_documents"])
    documenti_mancanti = list(documenti_richiesti - documenti_forniti)

    return {
        "valido": len(documenti_mancanti) == 0,
        "documenti_forniti": list(documenti_forniti),
        "documenti_mancanti": documenti_mancanti
    }


def verifica_limite_budget(sinistro: dict, requisito: dict) -> dict:
    """
    Regola 2: Verifica se il danno stimato rientra nel budget per la revisione automatica.
    """
    if "estimated_damage" not in sinistro or not isinstance(sinistro["estimated_damage"], (int, float)):
        raise ErroreTipoNonValido("Il campo 'estimated_damage' deve essere un numero valido.")
    if "automatic_review_max" not in requisito or not isinstance(requisito["automatic_review_max"], (int, float)):
        raise ErroreTipoNonValido("Il campo 'automatic_review_max' deve essere un numero valido.")

    danno_stimato = sinistro["estimated_damage"]
    massimo_automatico = requisito["automatic_review_max"]

    return {
        "rientra_nel_budget": danno_stimato <= massimo_automatico,
        "danno_stimato": danno_stimato,
        "massimo_automatico": massimo_automatico
    }


def verifica_polizza_attiva(polizza: dict) -> dict:
    """
    Regola 3: Verifica se il cliente è coperto da una polizza attualmente attiva.
    """
    if "active" not in polizza or not isinstance(polizza["active"], bool):
        raise ErroreDatiMancanti("La polizza non contiene un campo booleano 'active' valido.")

    return {
        "attiva": polizza["active"]
    }


def verifica_tipo_copertura(sinistro: dict, polizza: dict) -> dict:
    """
    Regola 4: Verifica se il tipo di danno dichiarato è coperto dal tipo di polizza sottoscritta.
    """
    if "loss_type" not in sinistro or not isinstance(sinistro["loss_type"], str):
        raise ErroreDatiMancanti("Il sinistro non contiene un campo 'loss_type' valido.")
    if "coverage" not in polizza or not isinstance(polizza["coverage"], str):
        raise ErroreDatiMancanti("La polizza non contiene un campo 'coverage' valido.")

    corrisponde = sinistro["loss_type"] == polizza["coverage"]

    return {
        "corrisponde": corrisponde,
        "tipo_danno_sinistro": sinistro["loss_type"],
        "copertura_polizza": polizza["coverage"]
    }


def verifica_limiti_polizza(sinistro: dict, polizza: dict) -> dict:
    """
    Regola 5: Verifica che l'importo sia superiore alla franchigia (minimo) e inferiore al massimale (massimo).
    """
    if "estimated_damage" not in sinistro or not isinstance(sinistro["estimated_damage"], (int, float)):
        raise ErroreTipoNonValido("Il campo 'estimated_damage' deve essere un numero.")
    
    for chiave in ["deductible", "max_claim"]:
        if chiave not in polizza or not isinstance(polizza[chiave], (int, float)):
            raise ErroreTipoNonValido(f"La polizza non contiene il campo numerico '{chiave}'.")

    danno = sinistro["estimated_damage"]
    franchigia = polizza["deductible"]
    massimale = polizza["max_claim"]

    sopra_minimo = danno > franchigia
    sotto_massimo = danno <= massimale

    return {
        "valido": sopra_minimo and sotto_massimo,
        "sopra_franchigia": sopra_minimo,
        "sotto_massimale": sotto_massimo,
        "franchigia": franchigia,
        "massimale": massimale
    }


# --- ORCHESTRATORE PRINCIPALE ---

def valuta_sinistro(sinistro: dict, polizze: list, requisiti: list, clienti: list = None) -> dict:
    """
    Esegue la sequenza controllata di tutte le regole gestendo qualsiasi eccezione nei dati.
    """
    id_sinistro = sinistro.get("claim_id", "SCONOSCIUTO")

    try:
        # Recupero dati associati
        polizza = next((p for p in polizze if p.get("policy_id") == sinistro.get("policy_id")), None)
        requisito = next((r for r in requisiti if r.get("loss_type") == sinistro.get("loss_type")), None)

        if not polizza:
            return {"id_sinistro": id_sinistro, "stato": "RIFIUTATO", "motivo": f"Polizza {sinistro.get('policy_id')} non trovata."}
        if not requisito:
            return {"id_sinistro": id_sinistro, "stato": "RIFIUTATO", "motivo": f"Nessun requisito trovato per il tipo di danno '{sinistro.get('loss_type')}'."}

        # Recupero facoltativo del cliente associato alla polizza
        cliente = None
        if clienti and "customer_id" in polizza:
            cliente = next((c for c in clienti if c.get("customer_id") == polizza["customer_id"]), None)

        # Esecuzione Regola 3: Polizza Attiva
        risultato_attiva = verifica_polizza_attiva(polizza)
        if not risultato_attiva["attiva"]:
            return {"id_sinistro": id_sinistro, "stato": "RIFIUTATO", "motivo": "La polizza non è attiva."}

        # Esecuzione Regola 4: Tipo di Copertura
        risultato_copertura = verifica_tipo_copertura(sinistro, polizza)
        if not risultato_copertura["corrisponde"]:
            return {
                "id_sinistro": id_sinistro, 
                "stato": "RIFIUTATO", 
                "motivo": f"Danno '{risultato_copertura['tipo_danno_sinistro']}' non coperto dalla polizza (copre: '{risultato_copertura['copertura_polizza']}')."
            }

        # Esecuzione Regola 5: Massimale e Franchigia
        risultato_limiti = verifica_limiti_polizza(sinistro, polizza)
        if not risultato_limiti["sopra_franchigia"]:
            return {
                "id_sinistro": id_sinistro, 
                "stato": "RIFIUTATO", 
                "motivo": f"Danno (€{sinistro['estimated_damage']}) inferiore o pari alla franchigia (€{risultato_limiti['franchigia']})."
            }
        if not risultato_limiti["sotto_massimale"]:
            return {
                "id_sinistro": id_sinistro, 
                "stato": "RIFIUTATO", 
                "motivo": f"Danno (€{sinistro['estimated_damage']}) superiore al massimale (€{risultato_limiti['massimale']})."
            }

        # Esecuzione Regole 1 & 6: Verifica Documenti
        risultato_documenti = verifica_documenti_richiesti(sinistro, requisito)
        if not risultato_documenti["valido"]:
            return {
                "id_sinistro": id_sinistro, 
                "stato": "IN_ATTESA", 
                "motivo": f"Documenti mancanti: {risultato_documenti['documenti_mancanti']}",
                "forniti": risultato_documenti["documenti_forniti"]
            }

        # Esecuzione Regola 2: Verifica Budget Approvazione Automatica
        risultato_budget = verifica_limite_budget(sinistro, requisito)
        if not risultato_budget["rientra_nel_budget"]:
            return {
                "id_sinistro": id_sinistro, 
                "stato": "REVISIONE_MANUALE", 
                "motivo": f"Importo (€{risultato_budget['danno_stimato']}) supera la soglia di approvazione automatica (€{risultato_budget['massimo_automatico']})."
            }

        risultato_finale = {
            "id_sinistro": id_sinistro, 
            "stato": "APPROVATO", 
            "motivo": "Tutti i requisiti deterministici sono stati soddisfatti con successo."
        }
        if cliente:
            risultato_finale["nome_cliente"] = cliente.get("name")
            
        return risultato_finale

    except ErroreValidazione as ev:
        return {"id_sinistro": id_sinistro, "stato": "ERRORE", "tipo_errore": type(ev).__name__, "messaggio": str(ev)}
    except Exception as e:
        return {"id_sinistro": id_sinistro, "stato": "ERRORE_SISTEMA", "messaggio": f"Errore generico non gestito: {str(e)}"}


# --- CARICAMENTO DEI FILE JSON 2 ED ESECUZIONE ---

if __name__ == "__main__":
    # Caricamento dati dai file json_2
    with open("claims_2.json", "r", encoding="utf-8") as f:
        sinistri = json.load(f)
    with open("policies_2.json", "r", encoding="utf-8") as f:
        polizze = json.load(f)
    with open("requirements_2.json", "r", encoding="utf-8") as f:
        requisiti = json.load(f)
    with open("customers_2.json", "r", encoding="utf-8") as f:
        clienti = json.load(f)

    print("--- RISULTATI VALUTAZIONE SINISTRI ---")
    for sinistro in sinistri:
        esito = valuta_sinistro(sinistro, polizze, requisiti, clienti)
        print(json.dumps(esito, indent=2, ensure_ascii=False))

: 